# Model Deployment Notebook

Deploy the trained fraud detection model to a SageMaker serverless endpoint.

This notebook handles:
1. **Model Selection** — Choose from trained models in the Model Registry
2. **Endpoint Creation** — Deploy to a serverless endpoint with custom inference handler
3. **Endpoint Testing** — Verify the endpoint works correctly
4. **Endpoint Management** — Update or delete endpoints

**Prerequisites:**
- Run `1_training_pipeline.ipynb` first to train a model
- Model must be registered in SageMaker Model Registry

**Custom Inference Handler:**
The endpoint uses a custom handler that automatically logs all predictions to Athena for drift monitoring.

**Environment:** SageMaker AI Notebook or local with AWS credentials.

## Setup

In [1]:
import os
import sys
import json
import boto3
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Find project root and add to path
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.config.config import (
    AWS_DEFAULT_REGION,
    SAGEMAKER_EXEC_ROLE,
    DATA_S3_BUCKET,
    ATHENA_DATABASE,
    ATHENA_OUTPUT_S3,
    MLFLOW_MODEL_NAME,
)
from src.utils.aws_utils import get_execution_role

# Initialize clients
region = AWS_DEFAULT_REGION
sm_client = boto3.client('sagemaker', region_name=region)
runtime_client = boto3.client('sagemaker-runtime', region_name=region)
s3_client = boto3.client('s3', region_name=region)

role = get_execution_role()

print(f'Region: {region}')
print(f'Role: {role}')
print(f'S3 Bucket: {DATA_S3_BUCKET}')
print(f'Athena DB: {ATHENA_DATABASE}')
print(f'MLflow Model: {MLFLOW_MODEL_NAME}')

Using SageMaker execution role from environment: arn:aws:iam::<ACCOUNT_ID>:role/fraud-detection-monitoring-SageMakerExecutionRole
Region: us-west-2
Role: arn:aws:iam::<ACCOUNT_ID>:role/fraud-detection-monitoring-SageMakerExecutionRole
S3 Bucket: fraud-detection-monitoring-data-<ACCOUNT_ID>
Athena DB: fraud_detection
MLflow Model: xgboost-fraud-detector


## Deployment Configuration

In [2]:
# Endpoint configuration
ENDPOINT_NAME = 'fraud-detector-endpoint'
ENDPOINT_CONFIG_NAME = f'{ENDPOINT_NAME}-config-{int(datetime.now().timestamp())}'
MODEL_PACKAGE_GROUP = 'fraud-detection'

# Serverless configuration
MEMORY_SIZE_MB = 2048  # 1024, 2048, 3072, 4096, 5120, 6144
MAX_CONCURRENCY = 10   # Max concurrent invocations

# Environment variables for custom inference handler
INFERENCE_ENV = {
    'ENABLE_ATHENA_LOGGING': 'true',
    'ATHENA_DATABASE': ATHENA_DATABASE,
    'ATHENA_OUTPUT_S3': ATHENA_OUTPUT_S3,
    'DATA_S3_BUCKET': DATA_S3_BUCKET,
    'ENDPOINT_NAME': ENDPOINT_NAME,
    'MLFLOW_TRACKING_URI': os.getenv('MLFLOW_TRACKING_URI', ''),
    'MLFLOW_MODEL_NAME': MLFLOW_MODEL_NAME,
    'MLFLOW_RUN_ID': 'pipeline',
    'MODEL_VERSION': 'v1',
}

print('Endpoint Configuration:')
print(f'  Name: {ENDPOINT_NAME}')
print(f'  Memory: {MEMORY_SIZE_MB} MB')
print(f'  Max Concurrency: {MAX_CONCURRENCY}')
print(f'\nAthena Logging: {INFERENCE_ENV["ENABLE_ATHENA_LOGGING"]}')

Endpoint Configuration:
  Name: fraud-detector-endpoint
  Memory: 2048 MB
  Max Concurrency: 10

Athena Logging: true


## 1. Select Model from Registry

Choose which model version to deploy.

In [3]:
# List available models in the registry
print(f'Available models in package group: {MODEL_PACKAGE_GROUP}\n')

try:
    response = sm_client.list_model_packages(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelApprovalStatus='Approved',
        SortBy='CreationTime',
        SortOrder='Descending',
        MaxResults=10
    )
    
    models = response.get('ModelPackageSummaryList', [])
    
    if not models:
        print('⚠️  No approved models found!')
        print('   Run 1_training_pipeline.ipynb first to train a model.')
    else:
        print(f'Found {len(models)} approved model(s):\n')
        for i, model in enumerate(models, 1):
            arn = model['ModelPackageArn']
            version = model.get('ModelPackageVersion', 'N/A')
            status = model['ModelApprovalStatus']
            created = model['CreationTime'].strftime('%Y-%m-%d %H:%M:%S')
            
            print(f'{i}. Version {version} ({status})')
            print(f'   Created: {created}')
            print(f'   ARN: {arn}')
            print()
        
        # Use the latest approved model
        LATEST_MODEL_ARN = models[0]['ModelPackageArn']
        LATEST_MODEL_VERSION = models[0].get('ModelPackageVersion', 1)
        
        print(f'✓ Will deploy latest model: {LATEST_MODEL_ARN}')
        print(f'✓ Model version: {LATEST_MODEL_VERSION}')
        
except sm_client.exceptions.ResourceNotFound:
    print(f'❌ Model package group "{MODEL_PACKAGE_GROUP}" not found!')
    print('   Create it by running the training pipeline first.')
except Exception as e:
    print(f'Error listing models: {e}')

Available models in package group: fraud-detection

Found 3 approved model(s):

1. Version 3 (Approved)
   Created: 2026-06-25 16:50:20
   ARN: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:model-package/fraud-detection/3

2. Version 2 (Approved)
   Created: 2026-06-25 16:23:40
   ARN: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:model-package/fraud-detection/2

3. Version 1 (Approved)
   Created: 2026-06-25 05:13:35
   ARN: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:model-package/fraud-detection/1

✓ Will deploy latest model: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:model-package/fraud-detection/3
✓ Model version: 3


## 2. Create SageMaker Model with Inference Code

Package the model artifact with the custom inference handler.

In [4]:
# Get model details from registry
model_desc = sm_client.describe_model_package(ModelPackageName=LATEST_MODEL_ARN)
model_data_url = model_desc['InferenceSpecification']['Containers'][0]['ModelDataUrl']
image_uri = model_desc['InferenceSpecification']['Containers'][0]['Image']

print(f'Model artifact: {model_data_url}')
print(f'Container image: {image_uri}')

# Update INFERENCE_ENV with actual model version
INFERENCE_ENV['MODEL_VERSION'] = f'v{LATEST_MODEL_VERSION}'
INFERENCE_ENV['MLFLOW_RUN_ID'] = f'model-registry-v{LATEST_MODEL_VERSION}'

print(f'\nModel version for inference logging: {INFERENCE_ENV["MODEL_VERSION"]}')
print(f'MLflow Run ID: {INFERENCE_ENV["MLFLOW_RUN_ID"]}')

# Create model name
model_name = f'{ENDPOINT_NAME}-model-{int(datetime.now().timestamp())}'

print(f'\nCreating SageMaker model: {model_name}...')

try:
    create_model_response = sm_client.create_model(
        ModelName=model_name,
        PrimaryContainer={
            'Image': image_uri,
            'ModelDataUrl': model_data_url,
            'Environment': INFERENCE_ENV
        },
        ExecutionRoleArn=role
    )
    
    print(f'✓ Model created: {model_name}')
    print(f'  ARN: {create_model_response["ModelArn"]}')
    print(f'  Version: {INFERENCE_ENV["MODEL_VERSION"]}')
    
except Exception as e:
    print(f'❌ Failed to create model: {e}')
    raise

Model artifact: s3://sagemaker-us-west-2-<ACCOUNT_ID>/fraud-detection/training/output/pipelines-bji6z6669fa3-TrainModel-IkJJq3fe6W/output/model.tar.gz
Container image: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1

Model version for inference logging: v3
MLflow Run ID: model-registry-v3

Creating SageMaker model: fraud-detector-endpoint-model-1782406505...


✓ Model created: fraud-detector-endpoint-model-1782406505
  ARN: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:model/fraud-detector-endpoint-model-1782406505
  Version: v3


## 3. Create Endpoint Configuration

In [5]:
print(f'Creating endpoint configuration: {ENDPOINT_CONFIG_NAME}...')

try:
    create_config_response = sm_client.create_endpoint_config(
        EndpointConfigName=ENDPOINT_CONFIG_NAME,
        ProductionVariants=[
            {
                'VariantName': 'AllTraffic',
                'ModelName': model_name,
                'ServerlessConfig': {
                    'MemorySizeInMB': MEMORY_SIZE_MB,
                    'MaxConcurrency': MAX_CONCURRENCY
                }
            }
        ]
    )
    
    print(f'✓ Endpoint configuration created: {ENDPOINT_CONFIG_NAME}')
    print(f'  ARN: {create_config_response["EndpointConfigArn"]}')
    
except Exception as e:
    print(f'❌ Failed to create endpoint configuration: {e}')
    raise

Creating endpoint configuration: fraud-detector-endpoint-config-1782406505...


✓ Endpoint configuration created: fraud-detector-endpoint-config-1782406505
  ARN: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:endpoint-config/fraud-detector-endpoint-config-1782406505


## 4. Deploy Endpoint

Create or update the endpoint. This takes 5-10 minutes for serverless endpoints.

In [6]:
import time

# Check if endpoint already exists
endpoint_exists = False
try:
    existing = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_exists = True
    endpoint_status = existing['EndpointStatus']
    print(f'Endpoint {ENDPOINT_NAME} already exists (status: {endpoint_status})')
    
    if endpoint_status == 'Failed':
        print('\nDeleting failed endpoint...')
        sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
        print('Waiting for deletion to complete...', end='', flush=True)
        while True:
            try:
                sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
                print('.', end='', flush=True)
                time.sleep(5)
            except:
                print(' Done!')
                endpoint_exists = False
                break
    
except sm_client.exceptions.ClientError as e:
    if 'Could not find endpoint' in str(e):
        print(f'Endpoint {ENDPOINT_NAME} does not exist yet')
        endpoint_exists = False
    else:
        raise

# Create or update endpoint
if endpoint_exists:
    print(f'\nUpdating endpoint: {ENDPOINT_NAME}...')
    try:
        sm_client.update_endpoint(
            EndpointName=ENDPOINT_NAME,
            EndpointConfigName=ENDPOINT_CONFIG_NAME
        )
        print(f'✓ Endpoint update initiated')
    except Exception as e:
        print(f'❌ Failed to update endpoint: {e}')
        raise
else:
    print(f'\nCreating endpoint: {ENDPOINT_NAME}...')
    try:
        create_endpoint_response = sm_client.create_endpoint(
            EndpointName=ENDPOINT_NAME,
            EndpointConfigName=ENDPOINT_CONFIG_NAME
        )
        print(f'✓ Endpoint creation initiated')
        print(f'  ARN: {create_endpoint_response["EndpointArn"]}')
    except Exception as e:
        print(f'❌ Failed to create endpoint: {e}')
        raise

# Monitor deployment
print(f'\nMonitoring deployment (this takes 5-10 minutes)...')
print(f'Checking status every 30 seconds:\n')

start_time = time.time()
while True:
    response = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = response['EndpointStatus']
    elapsed = int(time.time() - start_time)
    
    print(f'[{elapsed}s] Status: {status}', end='')
    
    if status == 'InService':
        print(' ✓ READY!')
        print(f'\n✓ Endpoint is live and ready for inference!')
        print(f'  Name: {ENDPOINT_NAME}')
        print(f'  ARN: {response["EndpointArn"]}')
        break
    elif status == 'Failed':
        print(' ❌ FAILED!')
        if 'FailureReason' in response:
            print(f'\nFailure reason: {response["FailureReason"]}')
        break
    else:
        print()
        time.sleep(30)

Endpoint fraud-detector-endpoint already exists (status: InService)

Updating endpoint: fraud-detector-endpoint...


✓ Endpoint update initiated

Monitoring deployment (this takes 5-10 minutes)...
Checking status every 30 seconds:

[0s] Status: Updating


[30s] Status: Updating


[60s] Status: Updating


[90s] Status: Updating


[120s] Status: Updating


## 5. Test Endpoint

Send a test prediction to verify the endpoint works.

In [ ]:
# Test payload (30 features)
test_features = {
    'transaction_hour': 14,
    'transaction_day_of_week': 3,
    'transaction_amount': 125.50,
    'transaction_type_code': 1,
    'customer_age': 35,
    'customer_gender': 1,
    'customer_tenure_months': 24,
    'account_age_days': 730,
    'distance_from_home_km': 5.2,
    'distance_from_last_transaction_km': 2.1,
    'time_since_last_transaction_min': 120,
    'online_transaction': 1,
    'international_transaction': 0,
    'high_risk_country': 0,
    'merchant_category_code': 5411,
    'merchant_reputation_score': 0.85,
    'chip_transaction': 1,
    'pin_used': 1,
    'card_present': 1,
    'cvv_match': 1,
    'address_verification_match': 1,
    'num_transactions_24h': 2,
    'num_transactions_7days': 8,
    'avg_transaction_amount_30days': 95.30,
    'max_transaction_amount_30days': 250.00,
    'velocity_score': 0.3,
    'recurring_transaction': 0,
    'previous_fraud_incidents': 0,
    'credit_limit': 5000.0,
    'available_credit_ratio': 0.75
}

print('Sending test prediction request...')
print(f'Test features: {json.dumps(test_features, indent=2)}\n')

try:
    response = runtime_client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(test_features)
    )
    
    result = json.loads(response['Body'].read().decode())
    
    print('✓ Prediction successful!')
    print(f'\nResult: {json.dumps(result, indent=2)}')
    
    if 'prediction' in result:
        pred = result['prediction']
        print(f'\nFraud Detection Result:')
        print(f'  Prediction: {"FRAUD" if pred == 1 else "NOT FRAUD"}')
        if 'probability' in result:
            print(f'  Probability: {result["probability"]:.4f}')
    
except Exception as e:
    print(f'❌ Prediction failed: {e}')

## 6. Endpoint Management

Utility functions for managing the endpoint.

In [ ]:
def get_endpoint_status(endpoint_name):
    """Get current endpoint status."""
    try:
        response = sm_client.describe_endpoint(EndpointName=endpoint_name)
        print(f'Endpoint: {endpoint_name}')
        print(f'Status: {response["EndpointStatus"]}')
        print(f'Config: {response["EndpointConfigName"]}')
        print(f'ARN: {response["EndpointArn"]}')
        return response
    except Exception as e:
        print(f'Error: {e}')
        return None

def delete_endpoint(endpoint_name):
    """Delete an endpoint."""
    try:
        sm_client.delete_endpoint(EndpointName=endpoint_name)
        print(f'✓ Deleted endpoint: {endpoint_name}')
    except Exception as e:
        print(f'Error: {e}')

print('Management functions available:')
print('  - get_endpoint_status(endpoint_name)')
print('  - delete_endpoint(endpoint_name)')
print(f'\nExample: get_endpoint_status("{ENDPOINT_NAME}")')

## Next Steps

Now that your endpoint is deployed:

1. **Run Inference** → Go to `3_inference_monitoring.ipynb` to:
   - Send predictions to the endpoint
   - Monitor inference performance
   - Detect data and model drift

2. **View Metrics** → Go to `4_governance_dashboard.ipynb` to:
   - View model performance over time
   - Analyze drift patterns
   - Generate governance reports

3. **Monitor Endpoint** → AWS Console:
   - CloudWatch metrics for invocations and latency
   - CloudWatch logs for detailed request/response logging
   - Athena for querying logged predictions